<a href="https://colab.research.google.com/github/brendanpshea/database_sql/blob/main/Database_12_DataIntegration.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Data Integration at Emerald City Clinic
## Database and SQL Through Pop Culture | Brendan Shea, PhD

Every database project so far has started with the data already inside a tidy SQLite file. Real projects almost never start that way. The data is scattered — a CSV from one department, a JSON API from another, an XML feed from a legacy system, a streaming pharmacy queue, even a web page that has to be scraped. The work of pulling that data together, cleaning it up, and getting it into a working warehouse is **data integration**.

This chapter walks through that work using a fictional pop-culture setting: **Emerald City Clinic**, a Wizard of Oz–themed medical facility on the outskirts of Oz. By the end, you will know how to acquire data from many sources, transform it with Pandas, load it into SQLite, troubleshoot the failures you will actually hit, and use AI tools wisely in the process.

## Learning Outcomes

By the end of this chapter, you will be able to:

1. Acquire data from CSV, JSON, XML, REST and OData, ODBC/JDBC, FTP, NFS, SSH, CIFS, RPC, SOAP, streaming sources, and scraped HTML.
2. Use Pandas to read source data, clean it, merge it, and load it into SQLite.
3. Distinguish ETL from ELT and decide which pattern fits a scenario.
4. Diagnose and fix the most common data acquisition failures: schema mismatch, encoding, permissions, driver/version issues, encryption key handling, timeouts, corruption, and syntax/runtime errors.
5. Explain where AI and modern tooling — prompt engineering, RAG, virtual data warehouses, ML libraries, RPA — help and where they introduce risk.

Keywords: data integration, ETL, ELT, CSV, JSON, XML, REST, OData, SOAP, ODBC, JDBC, streaming, scraping, Pandas, SQLite, prompt engineering, RAG, RPA

## Brendan's Lecture

In [ ]:
from IPython.display import YouTubeVideo
# Lecture video to be linked here.

## Introduction to Emerald City Clinic

The clinic's data team has six members, and the chapter's case studies stay with this cast:

- **Dorothy** — newly hired data engineer. She is leading the consolidation project that pulls every department's data into one SQLite warehouse.
- **The Scarecrow** — runs the records team. He delivers patient demographics as CSV exports.
- **The Tin Man** — owns lab systems. His department exposes lab results through a JSON REST API.
- **The Cowardly Lion** — handles security and compliance. He cares about access control, encryption keys, and certificates.
- **The Wizard** — the CIO. He sets strategic direction and signs off on the project.
- **Glinda** — the analytics lead. She consumes the consolidated warehouse for reporting.

The goal is one SQLite database that combines the Scarecrow's patients, the Tin Man's labs, the Wizard's legacy charts, and a steady stream of pharmacy refills.

## Generating the source data

Real source files come from real systems. For this chapter, a Python script produces a deterministic set of source files we can work with throughout the chapter — CSV, JSON, XML, a streaming feed, and a web page.

The same code lives in `data/emerald_city/generator.py` for reuse. The cell below reproduces the generator inline so the notebook runs end-to-end in Colab without any extra setup. Re-running the cell at any point gives you a fresh copy of the source files.

In [ ]:
# Inline copy of data/emerald_city/generator.py — produces all source files
# used throughout this chapter. Seeded so row counts and IDs are reproducible.
import csv, json, random, textwrap
from pathlib import Path

FIRST_NAMES = ["Dorothy", "Toto", "Glinda", "Aunt Em", "Uncle Henry", "Boq",
               "Nessa", "Fiyero", "Ozma", "Tip", "Polychrome", "Jack",
               "Mombi", "Jinjur", "Trot"]
LAST_NAMES = ["Gale", "Quadling", "Munchkin", "Gillikin", "Winkie",
              "Emerald", "Lurline", "Pingaree", "Pumperdink"]
TOWNS = ["Munchkinland", "Quadling Country", "Winkie Country",
         "Gillikin Country", "Emerald City"]
LAB_PANELS = ["CBC", "BMP", "CMP", "Lipid Panel", "TSH", "HbA1c", "Urinalysis"]
LAB_FLAGS = ["normal", "normal", "normal", "high", "low"]
CHART_DEPTS = ["Cardiology", "Endocrinology", "Pulmonology", "Family Medicine"]
PHARMACY_DRUGS = [("Lisinopril 10mg", 30), ("Metformin 500mg", 60),
                  ("Atorvastatin 20mg", 30), ("Levothyroxine 50mcg", 30),
                  ("Amoxicillin 500mg", 21)]


def generate(out_dir="emerald_city_data", seed=137):
    rng = random.Random(seed)
    out = Path(out_dir); out.mkdir(parents=True, exist_ok=True)

    patients = []
    for i in range(1, 26):
        patients.append({
            "patient_id": f"ECC{i:04d}",
            "first_name": rng.choice(FIRST_NAMES),
            "last_name": rng.choice(LAST_NAMES),
            "date_of_birth": f"{rng.randint(1935, 2020):04d}-{rng.randint(1,12):02d}-{rng.randint(1,28):02d}",
            "town": rng.choice(TOWNS),
            "primary_provider": f"Dr. {rng.choice(LAST_NAMES)}",
        })

    labs = []
    for p in patients:
        for _ in range(rng.randint(0, 3)):
            labs.append({
                "lab_id": f"LAB{len(labs) + 1:05d}",
                "patient_id": p["patient_id"],
                "panel": rng.choice(LAB_PANELS),
                "collected_on": f"2026-{rng.randint(1,5):02d}-{rng.randint(1,28):02d}",
                "result_flag": rng.choice(LAB_FLAGS),
                "ordering_provider": p["primary_provider"],
            })

    parts = ["<charts>"]
    for p in patients[:10]:
        parts.append(f'  <chart patient_id="{p["patient_id"]}">')
        parts.append(f"    <opened_on>{rng.randint(2010, 2024)}-0{rng.randint(1,9)}-1{rng.randint(0,9)}</opened_on>")
        parts.append(f"    <department>{rng.choice(CHART_DEPTS)}</department>")
        parts.append("    <note>Legacy chart imported from the Wizard system.</note>")
        parts.append("  </chart>")
    parts.append("</charts>")
    charts_xml = "\n".join(parts)

    pharmacy = []
    for _ in range(40):
        p = rng.choice(patients)
        drug, days = rng.choice(PHARMACY_DRUGS)
        pharmacy.append(json.dumps({
            "event_id": f"PH{len(pharmacy) + 1:05d}",
            "patient_id": p["patient_id"],
            "drug": drug,
            "days_supply": days,
            "filled_on": f"2026-05-{rng.randint(1,24):02d}",
        }))

    html = textwrap.dedent("""\
        <!doctype html><html><head><title>Emerald City Clinic Partner Directory</title></head>
        <body><h1>Partner Directory</h1>
        <table id="partners"><thead><tr><th>Partner</th><th>Specialty</th><th>Phone</th></tr></thead>
        <tbody>
          <tr><td>Tin Man Diagnostics</td><td>Lab</td><td>555-0101</td></tr>
          <tr><td>Scarecrow Records</td><td>Records</td><td>555-0102</td></tr>
          <tr><td>Lion Security</td><td>Compliance</td><td>555-0103</td></tr>
          <tr><td>Glinda Analytics</td><td>Analytics</td><td>555-0104</td></tr>
        </tbody></table></body></html>
    """)

    soap = textwrap.dedent("""\
        <?xml version="1.0"?>
        <soap:Envelope xmlns:soap="http://schemas.xmlsoap.org/soap/envelope/">
          <soap:Body><GetClinicHoursResponse>
            <Hours><Day>Monday</Day><Open>08:00</Open><Close>17:00</Close></Hours>
            <Hours><Day>Saturday</Day><Open>09:00</Open><Close>13:00</Close></Hours>
          </GetClinicHoursResponse></soap:Body>
        </soap:Envelope>
    """)

    with (out / "patients.csv").open("w", newline="", encoding="utf-8") as f:
        w = csv.DictWriter(f, fieldnames=list(patients[0].keys()))
        w.writeheader(); w.writerows(patients)
    (out / "labs.json").write_text(json.dumps(labs, indent=2), encoding="utf-8")
    (out / "charts.xml").write_text(charts_xml, encoding="utf-8")
    (out / "pharmacy_stream.txt").write_text("\n".join(pharmacy) + "\n", encoding="utf-8")
    (out / "partner_directory.html").write_text(html, encoding="utf-8")
    (out / "soap_response.xml").write_text(soap, encoding="utf-8")
    return out, patients, labs

OUT, PATIENTS, LABS = generate()
print(f"Wrote source files to {OUT.resolve()}")
print(f"  {len(PATIENTS)} patients, {len(LABS)} lab results")


## §6.1 Acquisition techniques and methods

This section covers the formats and protocols you will encounter when pulling data into a database. Most projects mix several at once, and Emerald City Clinic is no exception.

### File-based: CSV — the Scarecrow's records

CSV (comma-separated values) is the oldest, simplest format for tabular data. Every line is a row; every field is separated by a comma. The Scarecrow's records team exports a fresh `patients.csv` every morning.

Pandas can read it in one line:

In [ ]:
import pandas as pd

patients_df = pd.read_csv('emerald_city_data/patients.csv')
patients_df.head()

### File-based: JSON — the Tin Man's lab results

JSON is the format you use when the data has a hierarchy or when fields vary across records. Lab results, with their nested values and optional fields, are a natural fit. The Tin Man's API also publishes a daily JSON dump for offline use.

In [ ]:
labs_df = pd.read_json('emerald_city_data/labs.json')
labs_df.head()

### File-based: XML — the Wizard's legacy charts

XML is older than JSON but still common in healthcare, finance, and government systems. The Wizard's original chart system, set up decades ago in Oz, exports XML.

Python's standard library can parse it. Pandas can also read XML directly:

In [ ]:
charts_df = pd.read_xml('emerald_city_data/charts.xml')
charts_df.head()

### API-based: REST and OData

A **REST API** exposes resources at URLs and returns JSON or XML. Tin Man Diagnostics runs one at (imagined) `https://api.tinmandx.example.com/v1/labs`. You hit it with HTTP `GET` and parse the response.

**OData** is a convention layered on top of REST that adds a standard query syntax — `$filter`, `$select`, `$top` — so clients can do server-side filtering and pagination without learning each API's custom parameters. Many enterprise systems (Microsoft Dynamics, SAP) speak OData.

In a real notebook you would use `requests.get(url)`. For this lesson, we will read the Tin Man's daily JSON dump *as if* it came from the API — the data flow is the same.

In [ ]:
# Pretend this is a REST call. In production it would be:
#   response = requests.get('https://api.tinmandx.example.com/v1/labs')
#   labs_df = pd.DataFrame(response.json())
# For the chapter, we read the JSON dump the API would return.
labs_df = pd.read_json('emerald_city_data/labs.json')

# Equivalent of an OData $filter=result_flag eq 'high':
high_labs = labs_df[labs_df['result_flag'] == 'high']
high_labs.head()

### Driver-based: ODBC and JDBC

When the source is another database, you do not parse files — you connect through a **driver**. Two driver standards matter:

- **ODBC (Open Database Connectivity)** — a Microsoft-led standard, widely supported on Windows and many other platforms. C, Python, and R all have ODBC clients.
- **JDBC (Java Database Connectivity)** — the same idea, but for Java applications.

In Python, the most common path is **SQLAlchemy**, which sits above ODBC/JDBC drivers and offers a single API for many databases. Emerald City has an older patient-flow database that was built in a different SQLite file; Dorothy reads from it through SQLAlchemy.

In [ ]:
from sqlalchemy import create_engine, text

# Set up a tiny example "legacy" SQLite file so the demo is self-contained.
import sqlite3
with sqlite3.connect('legacy_flow.db') as con:
    con.execute('DROP TABLE IF EXISTS visits')
    con.execute('CREATE TABLE visits (visit_id INTEGER PRIMARY KEY, patient_id TEXT, visited_on TEXT)')
    con.executemany('INSERT INTO visits VALUES (?, ?, ?)',
                    [(1, 'ECC0001', '2026-05-01'),
                     (2, 'ECC0002', '2026-05-02'),
                     (3, 'ECC0001', '2026-05-10')])

# Now connect through SQLAlchemy as if it were any other database.
engine = create_engine('sqlite:///legacy_flow.db')
visits_df = pd.read_sql(text('SELECT * FROM visits'), engine)
visits_df

### Network protocols: FTP, NFS, SSH, CIFS, RPC, SOAP

Beyond HTTP, several older protocols still move data between systems. You will see them in firewall rules, vendor agreements, and audit logs.

| Protocol | What it does | Typical use at Emerald City |
| --- | --- | --- |
| **FTP / SFTP** | File transfer over the network | The Scarecrow's nightly CSV dropped on a shared server |
| **NFS** | Network file system (Unix) | A shared mount where the Tin Man writes lab PDFs |
| **CIFS / SMB** | Network file system (Windows) | A clinic-wide drive for scanned documents |
| **SSH** | Secure remote shell and file copy | Dorothy logs in to the records server to retrieve files |
| **RPC** | Remote procedure call | Older inter-system messages used by the Wizard's billing engine |
| **SOAP** | XML-based web services | The hours-of-operation feed that a partner uses |

The last one is worth a closer look. A **SOAP** response is XML wrapped in an `Envelope` and a `Body`:

In [ ]:
print(open('emerald_city_data/soap_response.xml').read())

### Streaming vs. non-streaming

A **non-streaming** source is a fixed file or query result — you read it all at once. The CSV and JSON examples above are non-streaming.

A **streaming** source delivers data continuously, one record at a time, often forever. The Tin Man's pharmacy partner pushes refill events to a queue, and Dorothy's loader reads them one at a time.

Python supports the streaming pattern naturally with file iteration: each line of `pharmacy_stream.txt` is one event.

In [ ]:
import json

count_by_drug = {}
with open('emerald_city_data/pharmacy_stream.txt') as f:
    for line in f:                         # one event at a time
        event = json.loads(line)
        count_by_drug[event['drug']] = count_by_drug.get(event['drug'], 0) + 1

for drug, n in sorted(count_by_drug.items(), key=lambda kv: -kv[1]):
    print(f"{n:>3d}  {drug}")

### Web scraping

When data lives in a web page but no API exposes it, scraping is the last resort. **BeautifulSoup** is the standard Python library for parsing HTML.

A few ethics rules before you ever scrape a real site:

- Check the site's terms of service.
- Respect `robots.txt`.
- Identify your scraper with a `User-Agent` header.
- Throttle requests so you do not overwhelm the server.

For practice, we scrape a local page the generator wrote — no live traffic involved.

In [ ]:
from bs4 import BeautifulSoup

soup = BeautifulSoup(open('emerald_city_data/partner_directory.html'), 'html.parser')
for row in soup.select('table#partners tbody tr'):
    cells_ = [c.get_text(strip=True) for c in row.find_all('td')]
    print(' | '.join(cells_))

## Transforming the data with Pandas

Once you have the data in memory, you usually need to clean it up before loading it into a database. Pandas does this well. The minimum vocabulary you need is small:

- `df.rename(columns={...})` — rename columns to match the target schema.
- `df['col'] = df['col'].astype(...)` — convert a column to the right type.
- `pd.to_datetime(df['col'])` — parse strings into proper dates.
- `df.merge(other, on='key')` — join two dataframes by a shared column.
- `df.to_sql('table', engine, if_exists='replace')` — write the dataframe to a database.

The Emerald City team needs to combine the Scarecrow's patient list with the Tin Man's labs so analysts can ask "which towns have the most high lab results?"

In [ ]:
# Read the two sources back in for the transformation example.
patients_df = pd.read_csv('emerald_city_data/patients.csv')
labs_df = pd.read_json('emerald_city_data/labs.json')

# Parse the date columns so we can filter on them later.
patients_df['date_of_birth'] = pd.to_datetime(patients_df['date_of_birth'])
labs_df['collected_on'] = pd.to_datetime(labs_df['collected_on'])

# Merge into a wider analytical view.
combined = labs_df.merge(patients_df, on='patient_id', how='left')
combined.head()

## Loading into SQLite: ETL vs. ELT

Two patterns describe how the cleaning work is split with the loading work.

- **ETL (Extract, Transform, Load).** Pull the data out of the source. Transform it in memory (Pandas, a script, a workflow tool). Load the cleaned result into the warehouse. The database only sees the polished version.
- **ELT (Extract, Load, Transform).** Pull the data out and load it into the warehouse in its raw shape. Transform it later, in SQL, using views or staging tables. The database sees both the raw and the cleaned versions.

ETL is older. It made sense when warehouses were expensive and small. ELT is newer and dominant in cloud warehouses, where storage is cheap and SQL is fast.

The Emerald City project uses ETL for the structured patient/lab combination and ELT for the streaming pharmacy data the analytics team prefers to inspect raw.

In [ ]:
from sqlalchemy import create_engine

# Build the target warehouse.
warehouse = create_engine('sqlite:///emerald_city.db')

# --- ETL example: transform in pandas, then load the clean result ---
clean = combined.rename(columns={
    'first_name': 'patient_first',
    'last_name':  'patient_last',
})
# We only want a small set of fields in the analytical table.
analytics = clean[['lab_id', 'patient_id', 'patient_first', 'patient_last',
                   'town', 'panel', 'result_flag', 'collected_on']]
analytics.to_sql('patient_lab_results', warehouse,
                 if_exists='replace', index=False)
print(f"Loaded {len(analytics)} rows into patient_lab_results")

In [ ]:
# --- ELT example: load the raw pharmacy stream, then transform in SQL ---
raw_pharmacy = []
with open('emerald_city_data/pharmacy_stream.txt') as f:
    for line in f:
        raw_pharmacy.append(json.loads(line))
pd.DataFrame(raw_pharmacy).to_sql(
    'pharmacy_events_raw', warehouse, if_exists='replace', index=False
)

# Transform inside SQLite with a view.
with warehouse.connect() as con:
    con.execute(text("DROP VIEW IF EXISTS pharmacy_by_drug"))
    con.execute(text("""
        CREATE VIEW pharmacy_by_drug AS
        SELECT drug, COUNT(*) AS refill_count, SUM(days_supply) AS total_days
        FROM pharmacy_events_raw
        GROUP BY drug
        ORDER BY refill_count DESC
    """))

pd.read_sql(text('SELECT * FROM pharmacy_by_drug'), warehouse)

## §6.2 Troubleshooting common data acquisition issues

Every integration project hits the same handful of failures. Knowing them by sight cuts hours off the debugging.

### Schema mismatch

The most common failure: the source file changes its column order or renames a column, and the loader silently shifts every value into the wrong field. Patient phone numbers end up in the date-of-birth column.

```python
# Loader assumes column order. The CSV gets re-ordered by mistake.
df = pd.read_csv('emerald_city_data/patients.csv', header=None,
                 names=['id', 'dob', 'first', 'last', 'town', 'provider'])  # WRONG order
```

**How to recognize it.** Values look like other values shifted one column over: ages of "Munchkinland," provider names that look like dates.

**Fix.** Never rely on column order. Always pass `header=0` (the default) and validate that the headers in the file match what you expected:

```python
df = pd.read_csv('emerald_city_data/patients.csv')
expected = {'patient_id', 'first_name', 'last_name', 'date_of_birth', 'town', 'primary_provider'}
missing = expected - set(df.columns)
assert not missing, f'Missing columns: {missing}'
```

### Encoding

A file written on a Windows machine in Western Europe arrives as Windows-1252. Python defaults to UTF-8 and rejects the bytes:

```python
# Throws: UnicodeDecodeError: 'utf-8' codec can't decode byte 0x96 in position 137
df = pd.read_csv('partner_export.csv')  # source written in cp1252
```

**How to recognize it.** A `UnicodeDecodeError`, often with a specific byte and position.

**Fix.** Tell Pandas the encoding:

```python
df = pd.read_csv('partner_export.csv', encoding='cp1252')
```

If you do not know the encoding, the `chardet` library can guess. When you control the upstream system, ask them to write UTF-8 and never look back.

### Permissions

The loader cannot read the source file:

```python
PermissionError: [Errno 13] Permission denied: '/mnt/scarecrow_share/patients.csv'
```

**How to recognize it.** `PermissionError` with a specific path.

**Fix.** Check four things, in order:

1. Does the file path exist? (`os.path.exists` will tell you.)
2. Does the *current user* have read access? (`ls -l` on Linux; *Properties → Security* on Windows.)
3. Is the share mounted with the right credentials?
4. Has someone rotated the service account password?

This is usually an ops problem, not a code problem. Loop in whoever owns the share before changing the code.

### Driver and version issues

Pandas refuses to write to SQLite:

```
ImportError: SQLAlchemy required for SQL support
```

or

```
DatabaseError: ... module 'sqlite3' has no attribute 'XYZ'
```

**How to recognize it.** Import errors or attribute errors that mention the driver itself.

**Fix.** Pin versions explicitly:

```bash
pip install --upgrade pandas sqlalchemy
```

Most "it works on my machine" failures come down to a driver version skew between development and production. Treat the driver as part of your application's dependencies, not part of the environment.

### Encryption keys in acquisition

The Tin Man's API responses are encrypted at rest. The decryption key is stored in a separate file on the loader's machine. If the key file is missing, the loader fails:

```
FileNotFoundError: [Errno 2] No such file or directory: '/etc/ecc/tinman.key'
```

or worse:

```
ValueError: MAC check failed
```

The first message is a missing key file. The second is the wrong key file. Both are common when promoting code from dev to prod and forgetting to deploy the keys.

**Fix.**

- Store keys in a secrets manager (cloud provider secret store, HashiCorp Vault), not on disk.
- Document key rotation as a step in the deployment checklist.
- Cross-reference Chapter 10's discussion of BYOK and KYOK — those choices decide who can recover from this kind of failure.

### Timeouts and corruption

A long file transfer gets cut off. The loader thinks the file is complete but it ends mid-record:

```python
json.decoder.JSONDecodeError: Expecting ',' delimiter: line 487 column 12 (char 14322)
```

**How to recognize it.** Parse errors that point to a position far into the file.

**Fix.**

1. Compare the file's size to the upstream size, or use a checksum (`md5sum`, `sha256sum`).
2. If they do not match, retry the transfer.
3. If they do match, the upstream file itself may be corrupt — escalate.

A robust integration pipeline always validates the transfer before parsing. The cheapest validation is a checksum check.

### Syntax and runtime errors

Two kinds of errors show up most:

- **Syntax errors** in SQL strings constructed with f-strings, especially when a value contains a quote. The fix is to use parameterized queries instead of string concatenation. Chapter 10's SQL-injection discussion is the security version of the same lesson.
- **Runtime errors** like `KeyError` when a column the loader expects is missing, or `ValueError` when a date cannot be parsed.

The best defense is a thin **validation step** between extract and transform: confirm the expected columns are there, sample-check a few values, and raise a clear error if something is off. A loud failure at validation is much cheaper than a silent failure that corrupts the warehouse.

## §6.3 Emerging tech and AI in data integration

AI tooling has become a routine part of data integration work. It is not magic, but it is genuinely useful when applied carefully.

### Where AI helps in this workflow

The honest answer is "drafting." AI coding assistants are good at:

- Writing the first pass of a parser for an unfamiliar format.
- Generating a regex that matches a description.
- Drafting a schema migration from a description of the new fields.
- Summarizing the API documentation for a system you have never used.

Dorothy uses an assistant on roughly half her work. The other half — deciding what to build, what to validate, and what to discard — is hers.

### Prompt engineering and human-in-the-loop

**Prompt engineering** is the skill of writing AI requests that produce useful results. A weak prompt: "write a parser for the Wizard's XML." A stronger prompt: "Here is a sample XML file with three records. Write a Python function `parse_charts(path)` that returns a list of dicts with keys `patient_id`, `opened_on`, `department`. Use `xml.etree.ElementTree`."

**Human-in-the-loop** is the matching discipline on the human side. The AI drafts; the engineer reads, runs tests, and only then commits. Dorothy never trusts AI-generated SQL without running it on test data first.

### Hallucinations and verification

AI assistants confidently make things up. They will reference functions that do not exist, columns that were never in the schema, and behavior the underlying library does not actually have.

A realistic example: Dorothy asks the assistant for a query to summarize labs by patient. The assistant returns:

```sql
SELECT patient_id, lab_summary(panel, result_flag) AS summary
FROM labs
GROUP BY patient_id;
```

`lab_summary` is not a real function. The assistant invented it from the shape of the question. Without verification, the query would have hit the database and failed — or worse, succeeded against a stub function and returned nonsense.

The rule: nothing AI-generated goes into the pipeline until a human has run it on real data and validated the result.

### Retrieval-augmented generation (RAG)

**Retrieval-augmented generation** combines a search system with a language model. Instead of asking the model to recall facts from training, the system first searches a trusted corpus, then asks the model to answer using only the retrieved passages.

For Emerald City, RAG lets clinic staff ask questions like "what are the most common high-flag panels this month?" The system retrieves the relevant rows from the warehouse, passes them to the language model, and the model writes a plain-English answer grounded in the retrieved data.

RAG does not eliminate hallucinations, but it makes them much rarer — the model has the actual data in front of it.

### ML libraries

A few libraries come up so often they are worth recognizing by name:

- **pandas** — the standard tool for cleaning and reshaping tabular data, used throughout this chapter.
- **scikit-learn** — classical machine learning models: regression, classification, clustering. The most common library for "predict X from these columns."
- **PyTorch** and **TensorFlow** — deep-learning frameworks for neural networks. Used heavily in image, audio, and language work.

Pandas is data prep; the others are model training. The two halves run on the same data but solve different problems.

### Virtual data warehouses

A **virtual data warehouse** presents many underlying sources as a single queryable warehouse — but it does not actually copy the data. Queries are translated and pushed down to the underlying systems.

Compare with what Dorothy built: a *materialized* warehouse where data is physically copied into SQLite. Virtual warehouses are great when the source systems are already fast and copying the data is expensive or non-compliant; materialized warehouses are simpler and faster for analytics.

Most large organizations end up with both — some data physically consolidated, some left in place behind a virtual layer.

### Robotic process automation (RPA)

**RPA** automates the routine click-and-type work that runs around a database — pulling a daily report out of a vendor portal, renaming files, emailing dashboards. It is less glamorous than ML but often more useful.

Emerald City uses an RPA bot to grab a partner's nightly CSV from a portal that has no API, drop the file onto the shared NFS mount, and trigger the loader. The bot has saved Dorothy hours every week and replaced a fragile manual process.

## Lab activity: extend the pipeline

Now it is your turn. Add a pharmacy CSV to the Emerald City pipeline.

1. Write a Python script that produces `emerald_city_data/pharmacy_partner.csv` with columns `patient_id`, `drug`, `days_supply`, `filled_on`. Use the generator function above as a starting point.
2. Read your CSV with Pandas.
3. Validate the columns. Raise a clear error if any expected column is missing.
4. Convert `filled_on` to a real date.
5. Merge it with the patient table on `patient_id`.
6. Load the merged result into a new SQLite table called `pharmacy_partner_loaded`.
7. Write a SQL query (against the warehouse) that returns the number of refills per town.

Bonus: deliberately corrupt your CSV (delete a column header) and verify that your validation step catches it before the bad data reaches the warehouse.

## Chapter Summary

Use this list as a self-check. After working through the chapter, you should be able to do each of the following:

- You can pull data from CSV, JSON, XML, REST and OData APIs, ODBC/JDBC drivers, network protocols, streaming sources, and scraped HTML.
- You can use Pandas to read, clean, merge, and load data into SQLite.
- You can pick between ETL and ELT for a given scenario.
- You can diagnose and recover from schema mismatch, encoding, permissions, driver/version, encryption key, timeout/corruption, and syntax/runtime failures.
- You can describe how AI tools help with integration work and what their limits are.

## Practice with the Loop of the Recursive Dragon

Sharpen what you just learned with a chapter-matched review set in the **Loop of the Recursive Dragon** — an adaptive review game with multiple question types and RPG-style mechanics, built for this book.

[**Launch the Chapter 12 review set →**](https://brendanpshea.github.io/LotRD/?set=database_12_data_integration.json)

## Glossary

Use this list as a quick review sheet for the chapter.

### Acquisition formats and protocols

- **ETL (Extract, Transform, Load)** — A data-integration pattern in which data is cleaned in memory before it is loaded into the warehouse.
- **ELT (Extract, Load, Transform)** — A data-integration pattern in which data is loaded into the warehouse in its raw shape and transformed later with SQL.
- **CSV** — A plain-text tabular format with one row per line and fields separated by commas.
- **XML** — A self-describing markup format that uses tags to represent structured data.
- **REST** — An architectural style for web APIs that exposes resources at URLs over HTTP.
- **OData** — A convention layered on REST that adds standard query syntax such as `$filter`, `$select`, and `$top`.
- **SOAP** — An older XML-based web-service protocol that wraps every message in an `Envelope` and a `Body`.
- **RPC (Remote Procedure Call)** — A pattern in which a client invokes a procedure on a remote system as if it were local.
- **ODBC** — A driver standard that lets applications connect to many database systems through one API.
- **JDBC** — The same idea as ODBC, but for Java applications.
- **FTP / SFTP** — A protocol for transferring files between systems; SFTP adds encryption.
- **NFS** — A network file system common in Unix environments.
- **CIFS / SMB** — A network file system common in Windows environments.
- **SSH** — A protocol for secure remote shell access and file copying.
- **Streaming source** — A data source that delivers records continuously, one at a time.
- **Non-streaming source** — A data source that delivers a fixed, finite result, such as a file.
- **Web scraping** — Extracting data from a web page by parsing its HTML.

### Troubleshooting

- **Schema mismatch** — A failure in which the source file's columns no longer match what the loader expects.
- **Character encoding** — The mapping between bytes and characters; a mismatch causes decoding errors.
- **Driver version mismatch** — A failure caused by incompatible versions of a database driver between systems.

### Emerging tech and AI

- **Prompt engineering** — The skill of phrasing AI requests so they produce useful results.
- **Retrieval-augmented generation (RAG)** — A pattern that combines a search step with a language model so answers are grounded in a trusted corpus.
- **Hallucination** — An AI response that confidently states something untrue, such as a non-existent function or column.
- **Human-in-the-loop** — A workflow in which a human reviews and approves AI-generated work before it takes effect.
- **Virtual data warehouse** — A warehouse that exposes many underlying sources as one queryable system without physically copying their data.
- **Robotic process automation (RPA)** — Software that automates routine click-and-type work around existing systems.
- **Data poisoning** — Cross-reference to Chapter 10: an attack that corrupts training or pipeline data to skew downstream behavior.